# 01장. NEIS API와 JSON

| 오늘의 질문 | 예상 시간 |
|---|---:|
| 우리 학교 급식은 어디에서 올까? | 1회차 후반 · 약 110분 |


## 이 장에서 배울 내용

- API의 요청과 응답을 식당 주문에 비유해 설명할 수 있다.
- JSON에서 키와 값을 구분할 수 있다.
- 학교명보다 학교 코드가 정확한 식별값인 이유를 말할 수 있다.


## 생각 열기

사람은 학교 급식표를 화면에서 읽지만 프로그램은 정해진 주소와 조건으로 데이터를 요청합니다. NEIS API에 학교 코드와 날짜를 보내고, 돌아온 JSON에서 메뉴를 찾습니다.


## 핵심 용어

| 용어 | 뜻 |
|---|---|
| **API** | 다른 서비스에 정해진 규칙으로 데이터를 요청하는 창구 |
| **요청** | 주소와 조건을 서버에 보내는 일 |
| **JSON** | 키와 값으로 데이터를 표현하는 형식 |
| **학교 코드** | 같은 이름의 학교를 구분하는 공식 식별값 |


## 개념 익히기


API는 급식실 주문 창구와 비슷합니다. ‘남악고등학교, 2026년 6월 24일 급식’을 정해진 양식으로 요청하면 서버가 정해진 양식의 응답을 줍니다.

JSON은 사물함에 이름표를 붙인 모습과 비슷합니다. **MLSV_YMD**라는 키에는 날짜 값이, **DDISH_NM**이라는 키에는 메뉴 값이 들어 있습니다. 키를 알면 긴 응답에서 필요한 값을 정확히 찾을 수 있습니다.


## 활동 전 생각


다음 작은 JSON에서 키와 값을 각각 표시해 보세요.

    {"school": "남악고등학교", "date": "20260624"}

키는 school과 date이고, 값은 남악고등학교와 20260624입니다.


## 예상하기

- 원본 행 수는 5이다.
- 첫 행의 키 목록에 MLSV_YMD와 DDISH_NM이 있다.


## 활동 1. 원본 JSON 행 관찰


### 코드 살펴보기


1. `raw_rows[0]`은 첫 번째 급식 기록 한 행을 고릅니다.<br>
2. `first_row.keys()`는 그 행에 들어 있는 모든 키를 가져옵니다.<br>
3. `first_row["MLSV_YMD"]`처럼 대괄호 안에 키를 쓰면 해당 값을 읽습니다.


In [ ]:
import sys
from pathlib import Path

current_folder = Path.cwd().resolve()
for candidate in (current_folder, *current_folder.parents):
    if (candidate / "jupyter_course" / "notebook_support.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError(
        "프로젝트 폴더를 찾지 못했습니다. 프로젝트 최상위 폴더에서 "
        r".\.venv\Scripts\python.exe -m notebook 명령으로 다시 시작하세요."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jupyter_course.notebook_support import course_setup

setup = course_setup(PROJECT_ROOT)
PROJECT_ROOT = setup["root"]
raw_rows = setup["rows"]
meal_df = setup["frame"]
data_source = setup["source"]
print("프로젝트 폴더:", PROJECT_ROOT)
print("데이터 출처:", data_source)
print("급식 행 수:", len(raw_rows))

first_row = raw_rows[0]
first_keys = sorted(first_row.keys())
print("첫 행의 키:")
print(first_keys)
print("날짜 값:", first_row["MLSV_YMD"])
print("메뉴 원문:", first_row["DDISH_NM"][:100] + "...")


### 결과 해석하기

키는 데이터의 이름표입니다. 메뉴 원문에는 HTML 줄바꿈과 괄호 속 번호가 아직 섞여 있습니다.


## 활동 2. 요청 조건을 사전으로 만들기


### 코드 살펴보기


1. `request_params`는 NEIS에 보낼 조건을 키와 값으로 묶은 사전입니다.<br>
2. `ATPT_OFCDC_SC_CODE`와 `SD_SCHUL_CODE`가 교육청과 학교를 구분합니다.<br>
3. `items()`는 요청 조건을 한 쌍씩 꺼내 확인하게 합니다.


In [ ]:
request_params = {
    "ATPT_OFCDC_SC_CODE": "Q10",
    "SD_SCHUL_CODE": "7140272",
    "MLSV_FROM_YMD": "20260624",
    "MLSV_TO_YMD": "20260630",
}
for key, value in request_params.items():
    print(f"{key} → {value}")


### 결과 해석하기

Q10은 전라남도교육청 코드, 7140272는 남악고등학교 코드입니다. 날짜는 YYYYMMDD 여덟 자리입니다.


## 활동 3. 실시간 우선·예비 자료 전환 연습


### 코드 살펴보기


1. `classroom_offline_demo`는 교실에서 연결 실패 상황을 재현합니다.<br>
2. `load_classroom_frame`은 실시간 조회가 실패하면 같은 기간의 예비 자료를 찾습니다.<br>
3. `try`와 `except NeisApiError`는 날짜가 맞지 않는 경우의 안내를 확인합니다.


In [ ]:
from jupyter_course.notebook_support import load_classroom_frame
from neis_meal_ai.neis import NeisApiError

def classroom_offline_demo(_school, _start, _end):
    raise NeisApiError("교실용 연결 실패 실험")

classroom_frame, classroom_source = load_classroom_frame(
    PROJECT_ROOT,
    fetcher=classroom_offline_demo,
)
print("선택된 데이터 출처:", classroom_source)
print("분석 가능한 급식 행 수:", len(classroom_frame))

try:
    load_classroom_frame(
        PROJECT_ROOT,
        start="20260101",
        end="20260102",
        fetcher=classroom_offline_demo,
    )
except NeisApiError as error:
    print("날짜가 겹치지 않을 때의 안내:", error)

chapter_result = {
    "chapter": "01",
    "source": classroom_source,
    "raw_rows": len(classroom_frame),
    "first_keys": first_keys,
}


### 결과 해석하기

실시간 조회가 실패해도 같은 학교·같은 기간의 예비 자료가 있을 때만 전환됩니다. 기간이 겹치지 않으면 조용히 다른 날짜를 쓰지 않고 정확한 안내를 보여 줍니다.


## 탐구 활동

LIVE_EXPERIMENT를 True로 바꾸면 공식 NEIS에서 남악고의 실제 급식을 요청합니다. 성공하면 실시간 자료를, 연결이 실패하면 같은 기간의 예비 자료를 사용합니다.

먼저 기본값으로 한 번 실행하세요. 그다음 표시된 값 하나만 바꾸고, 달라진 결과를 아래에 적습니다.


In [ ]:
LIVE_EXPERIMENT = False
if LIVE_EXPERIMENT:
    try:
        live_frame, live_source = load_classroom_frame(PROJECT_ROOT)
        print("실험 데이터 출처:", live_source)
        print(live_frame[["date", "menu_text"]].head().to_string(index=False))
    except (NeisApiError, ValueError) as error:
        print("NEIS 조회 안내:", error)
else:
    print("기본 학습 모드: 위에서 검증한 예비 자료 사용")


### 내가 본 변화

- 내가 바꾼 값:  
- 화면에서 달라진 것:  
- 내 설명:


## 확인 문제

1. API 요청에 학교 코드와 날짜가 필요한 이유는 무엇인가요?
2. JSON의 키와 값은 각각 어떤 역할을 하나요?
3. 실시간 API가 잠시 멈춰도 수업을 이어 갈 수 있는 이유는 무엇인가요?


## 정답과 해설


1. 어떤 학교의 어느 기간 자료인지 정확히 지정하기 위해서입니다.<br>
2. 키는 데이터의 이름표이고 값은 실제 내용입니다.<br>
3. 같은 구조의 공식 NEIS 예비 데이터 5행을 프로젝트에 포함했기 때문입니다.


## 핵심 정리

- API는 규칙이 있는 데이터 요청 창구다.
- JSON은 키와 값으로 구조를 표현한다.
- 남악고의 공식 코드는 Q10 / 7140272다.

### 다음 장

02장에서는 원본 문자열을 분석 가능한 표로 바꾸고 그래프로 읽습니다.


In [ ]:
import json
print("__CHAPTER_RESULT__=" + json.dumps(chapter_result, ensure_ascii=False))
